In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import os

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

save_dir = "data/xenium/processed"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
data_dir = "/bonn-epyc/projects/spatialTCR/20240719__094819__human_kidney_7_TCR"
!ls $data_dir

In [ ]:
adatas = []
metadatas = []
for folder in os.listdir(data_dir):
    if folder.startswith("output"):
        adata = sc.read_10x_h5(os.path.join(data_dir, folder, "cell_feature_matrix.h5"))
        adata.obs["sample"] = folder.split("-")[-1]
        cells = pd.read_parquet(
            os.path.join(data_dir, folder, "cells.parquet")
        ).set_index("cell_id")

        adata.obs.index = adata.obs.index + folder
        cells.index = cells.index + folder

        adatas.append(adata)
        metadatas.append(cells)

In [ ]:
adata = ad.concat(adatas, uns_merge="unique")
obs = pd.concat(metadatas, axis=0)
obs = obs.loc[adata.obs.index]

adata.obsm["spatial"] = obs.iloc[:, :2].values
adata.obs[obs.columns] = obs
print(adata.obs["cell_area"].isna().sum())

In [ ]:
adata.obs["slide"] = adata.obs["sample"].str.split("__").str[1]

In [ ]:
adata.obs["slide"].value_counts()

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
# add meta information?

In [ ]:
adata.obs["sample"].value_counts()

In [ ]:
# remove thymus sample
sample = "XETG00088__0029040__Region_1__20240719__095641"
ad_sub = adata[adata.obs["sample"] == sample].copy()
sc.pl.spatial(ad_sub, color="CD4", spot_size=15)

In [ ]:
adata = adata[adata.obs["sample"] != sample].copy()

## Basic quality control

In [ ]:
adata

In [ ]:
adata.obs.cell_area

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
def plot_qc(adata):
    fig, axs = plt.subplots(1, 4, figsize=(15, 4))

    axs[0].set_title("Total transcripts per cell")
    sns.histplot(
        adata.obs["total_counts"],
        kde=False,
        ax=axs[0],
    )

    axs[1].set_title("Unique transcripts per cell")
    sns.histplot(
        adata.obs["n_genes_by_counts"],
        kde=False,
        ax=axs[1],
    )

    axs[2].set_title("Area of segmented cells")
    sns.histplot(
        adata.obs["cell_area"],
        kde=False,
        ax=axs[2],
    )

    axs[3].set_title("Nucleus ratio")
    sns.histplot(
        adata.obs["nucleus_area"] / adata.obs["cell_area"],
        kde=False,
        ax=axs[3],
    )


plot_qc(adata)

In [ ]:
# filter by total transcripts, total unique transcripts, cell area

total_transcripts_limit = 500
unique_transcripts_limit = 100
cell_area_limit = 400

adata = adata[
    (adata.obs["total_counts"] <= total_transcripts_limit)
    & (adata.obs["n_genes_by_counts"] <= unique_transcripts_limit)
    & (adata.obs["cell_area"] <= cell_area_limit)
].copy()
plot_qc(adata)

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
plot_qc(adata)
adata.shape

## Add metadata

In [ ]:
disease_info = {
    "Control": [
        "XETG00088__0029040__Region_2__20240719__095641",
        "XETG00088__0029040__Region_3__20240719__095641",
        "XETG00088__0029041__Region_1__20240719__095642",
    ],
    "ANCA": [
        "XETG00088__0029040__Region_4__20240719__095642",
        "XETG00088__0029040__Region_5__20240719__095642",
        "XETG00088__0029040__Region_6__20240719__095642",
        "XETG00088__0029040__Region_7__20240719__095642",
        "XETG00088__0029041__Region_3__20240719__095642",
        "XETG00088__0029041__Region_4__20240719__095642",
        "XETG00088__0029041__Region_5__20240719__095642",
        "XETG00088__0029041__Region_6__20240719__095642",
        "XETG00088__0029041__Region_7__20240719__095642",
        "XETG00088__0029041__Region_8__20240719__095642",
    ],
    "BK-Virus": ["XETG00088__0029041__Region_2__20240719__095642"],
}

In [ ]:
for s in disease_info["Control"]:
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="CD3D", spot_size=15, cmap="Reds", vmax=1)

In [ ]:
adata.obs["condition"] = adata.obs["sample"]

for condition, samples in disease_info.items():
    mask = adata.obs["sample"].isin(samples)
    adata.obs.loc[mask, "condition"] = condition
adata.obs["condition"].value_counts()

In [ ]:
ad_sub = adata[adata.obs["condition"] == "BK-Virus"].copy()
sc.pl.spatial(ad_sub, color="CD4", spot_size=15)

In [ ]:
adata.obs["condition"].value_counts()

## Remove BK-Virus samples

In [ ]:
adata = adata[adata.obs["condition"] != "BK-Virus"].copy()
adata.obs["condition"].value_counts()

## Save anndata

In [ ]:
adata.write_h5ad(f"{save_dir}/01-kidney_tcr_qc.h5ad")